# MONZA × jpt — Análise gráfica dos resultados

Notebook para reproduzir os gráficos do experimento MONZA no cenário CNN/MNIST, comparando as defesas disponíveis e os detectores DistilBERT e MLP treinados em `state_dict`.

Defesas comparadas:
- `cc=2` — cluster por similaridade cosseno
- `cc=3` — cosseno + score ponderado
- `cc=6` — detector NLP/DistilBERT
- `cc=7` — detector MLP por features estatísticas do `state_dict`
- `cc=8` — MLP + validação pública, experimental e carregado somente quando `fpr_frr_results_8.csv` existir
- `cc=9` — BERT + MLP + verificação label flip por holdout limpo e camada final, carregado somente quando `fpr_frr_results_9.csv` existir

Os gráficos focam em:
- FPR/FRR por round
- média dos últimos 30 rounds
- trade-off FPR × FRR
- métricas gerais DistilBERT vs MLP
- FPR/recall por tipo de ataque no eval set do detector
- foco separado em `label flip`


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

JPT = Path.cwd().resolve()
MONZA = JPT / 'PFLlibMonza' / 'system'
OUT = JPT

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

print('JPT:', JPT)
print('MONZA:', MONZA)


## 1. Carregar CSVs (FPR/FRR por round)

A célula abaixo carrega todos os CSVs disponíveis. O `cc=8` é opcional para permitir rodar o notebook tanto no baseline antigo quanto no fluxo novo do HOWTO.


In [ ]:
def load_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    rename = {
        'False Positive Rate': 'FPR',
        'False Rejection Rate': 'FRR',
    }
    df = df.rename(columns=rename)

    required = {'Round', 'FPR', 'FRR'}
    missing_cols = required - set(df.columns)
    if missing_cols:
        raise ValueError(f'{path} sem colunas obrigatorias: {sorted(missing_cols)}')

    df = df[['Round', 'FPR', 'FRR']].copy()
    df = df[df['Round'].astype(str) != 'Round'].reset_index(drop=True)
    df = df.astype({'Round': int, 'FPR': float, 'FRR': float})

    # Se o CSV acumulou mais de um run, fica apenas com o ultimo trecho onde Round reiniciou.
    last_start = 0
    prev = -1
    for i, r in enumerate(df['Round'].values):
        if r < prev:
            last_start = i
        prev = r
    return df.iloc[last_start:].reset_index(drop=True)


DEFENSES = {
    'cc=2 (cluster cosseno)': MONZA / 'fpr_frr_results_2.csv',
    'cc=3 (cosseno+score)': MONZA / 'fpr_frr_results_3.csv',
    'cc=6 (NLP DistilBERT)': MONZA / 'fpr_frr_results_6.csv',
    'cc=7 (MLP+features)': MONZA / 'fpr_frr_results_7.csv',
    'cc=8 (MLP+validacao publica)': MONZA / 'fpr_frr_results_8.csv',
    'cc=9 (BERT+MLP+label flip check)': MONZA / 'fpr_frr_results_9.csv',
}

dfs = {}
missing = []
for name, path in DEFENSES.items():
    if path.exists():
        dfs[name] = load_csv(path)
    else:
        missing.append((name, path.name))

if not dfs:
    raise FileNotFoundError(f'Nenhum CSV fpr_frr_results_*.csv encontrado em {MONZA}')

for name, df in dfs.items():
    print(f'{name:36s} | {len(df):3d} rounds | FPR last: {df.FPR.iloc[-1]:.4f} | FRR last: {df.FRR.iloc[-1]:.4f}')

if missing:
    print('\nCSVs ausentes/ignorados:')
    for name, filename in missing:
        print(f'- {name}: {filename}')


## 2. Tabela de médias dos últimos 30 rounds


In [ ]:
rows = []
for name, df in dfs.items():
    last30 = df.tail(30)
    rows.append({
        'Defesa': name,
        'FPR_mean': last30['FPR'].mean(),
        'FPR_std': last30['FPR'].std(),
        'FRR_mean': last30['FRR'].mean(),
        'FRR_std': last30['FRR'].std(),
        'Score (FPR+FRR)': last30['FPR'].mean() + last30['FRR'].mean(),
    })
summary = pd.DataFrame(rows).sort_values('Score (FPR+FRR)').reset_index(drop=True)
summary


## 3. Curvas FPR e FRR por round


In [ ]:
COLORS = {
    'cc=2 (cluster cosseno)': '#1f77b4',
    'cc=3 (cosseno+score)': '#ff7f0e',
    'cc=6 (NLP DistilBERT)': '#2ca02c',
    'cc=7 (MLP+features)': '#d62728',
    'cc=8 (MLP+validacao publica)': '#9467bd',
    'cc=9 (BERT+MLP+label flip check)': '#8c564b',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True)

for name, df in dfs.items():
    color = COLORS.get(name, '#333333')
    axes[0].plot(df['Round'], df['FPR'], label=name, color=color, linewidth=2)
    axes[1].plot(df['Round'], df['FRR'], label=name, color=color, linewidth=2)

axes[0].set_title('False Positive Rate por round')
axes[0].set_xlabel('Round')
axes[0].set_ylabel('FPR')
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc='upper left', fontsize=9)
axes[0].set_ylim(-0.01, max(0.30, axes[0].get_ylim()[1]))

axes[1].set_title('False Rejection Rate por round')
axes[1].set_xlabel('Round')
axes[1].set_ylabel('FRR')
axes[1].grid(True, alpha=0.3)
axes[1].legend(loc='upper left', fontsize=9)
axes[1].set_ylim(-0.01, max(0.60, axes[1].get_ylim()[1]))

plt.tight_layout()
out = OUT / 'plot_fpr_frr_by_round.png'
plt.savefig(out, dpi=160, bbox_inches='tight')
out


## 4. Trade-off FPR × FRR (canto inferior-esquerdo = melhor defesa)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))

for _, row in summary.iterrows():
    name = row['Defesa']
    ax.scatter(
        row['FPR_mean'],
        row['FRR_mean'],
        s=220,
        color=COLORS.get(name, '#333333'),
        edgecolors='black',
        linewidths=1.2,
        label=name,
        zorder=5,
    )
    ax.annotate(name.split(' (')[0], (row['FPR_mean'], row['FRR_mean']), xytext=(10, 6), textcoords='offset points', fontsize=10, fontweight='bold')

max_fpr = max(float(summary['FPR_mean'].max()) * 1.25, 0.20)
max_frr = max(float(summary['FRR_mean'].max()) * 1.25, 0.35)
max_score = max(max_fpr + max_frr, 0.30)

for score in np.arange(0.10, max_score + 0.001, 0.10):
    xs = np.linspace(0, min(score, max_fpr), 50)
    ys = score - xs
    mask = ys <= max_frr
    if mask.any():
        ax.plot(xs[mask], ys[mask], '--', color='gray', alpha=0.25, lw=0.8)

ax.set_title('Trade-off FPR x FRR — medias dos ultimos 30 rounds')
ax.set_xlabel('FPR medio')
ax.set_ylabel('FRR medio')
ax.set_xlim(-0.01, max_fpr)
ax.set_ylim(-0.01, max_frr)
ax.grid(True, alpha=0.3)
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
out = OUT / 'plot_tradeoff_fpr_frr.png'
plt.savefig(out, dpi=160, bbox_inches='tight')
out


## 5. Métricas dos detectores no eval set in-sample (DistilBERT vs MLP)


In [ ]:
def load_json_if_exists(path: Path):
    if path.exists():
        with open(path) as f:
            return json.load(f)
    print(f'Arquivo ausente/ignorado: {path}')
    return None

nlp_metrics = load_json_if_exists(JPT / 'detector_monza_cnn_mnist' / 'metrics.json')
mlp_report = load_json_if_exists(JPT / 'detector_mlp_monza_cnn_mnist' / 'report.json')

metric_rows = {}
if nlp_metrics:
    default = nlp_metrics.get('default_argmax', {})
    metric_rows['NLP DistilBERT'] = {
        'Accuracy': default.get('accuracy', np.nan),
        'F1': default.get('f1', np.nan),
        'Precision': default.get('precision', np.nan),
        'Recall': default.get('recall', np.nan),
    }
if mlp_report:
    metrics = mlp_report.get('metrics', {})
    metric_rows['MLP features'] = {
        'Accuracy': metrics.get('accuracy', np.nan),
        'F1': metrics.get('f1', np.nan),
        'Precision': metrics.get('precision', np.nan),
        'Recall': metrics.get('recall', np.nan),
    }

metrics_compare = pd.DataFrame(metric_rows).T
metrics_compare


In [ ]:
if metrics_compare.empty:
    print('Sem artefatos de detector para plotar. Rode o Passo 2 do HOWTO primeiro.')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    metrics_compare.plot.bar(ax=ax, color=['#6c757d', '#2ca02c', '#1f77b4', '#ff7f0e'], width=0.7)
    ax.set_title('Detector training metrics — eval set (in-sample, otimista)')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(metrics_compare.index, rotation=0)
    ax.grid(alpha=0.3, axis='y')
    ax.legend(loc='lower center', ncol=4)
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8, padding=3)
    plt.tight_layout()
    out = OUT / 'plot_detector_metrics.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    display(out)


## 6. Recall por tipo de ataque — DistilBERT vs MLP

Para o MLP, o notebook usa `detector_mlp_monza_cnn_mnist/report.json`, gerado por `src/detector_mlp.py`.

Para o DistilBERT, o notebook usa `detector_monza_cnn_mnist/metrics.json -> default_argmax.by_type`, gerado por `src/detector.py`. Se o artefato for antigo e ainda não tiver `by_type`, o notebook usa o fallback histórico só para comparação.

Interpretação:
- `benign (FPR)` é falso positivo em clientes benignos; menor é melhor.
- Ataques maliciosos usam recall por tipo; maior é melhor.


In [ ]:
def by_type_rates_from_counts(by_type: dict) -> dict:
    values = {}
    if not by_type:
        return values
    for attack_type, item in by_type.items():
        total = item.get('total', 0)
        predicted = item.get('predicted_malicious', 0)
        rate = predicted / total if total else np.nan
        if attack_type == 'benign':
            values['benign (FPR)'] = rate
        else:
            values[attack_type] = rate
    return values

# Fallback do detector NLP antigo: valores recuperados do log impresso no experimento original.
nlp_recall_fallback = {
    'benign (FPR)': 12 / 720,
    'malicious_label': 5 / 81,
    'malicious_random': 69 / 70,
    'malicious_shuffle': 68 / 73,
    'malicious_zeros': 76 / 76,
}

by_type_sources = {}

nlp_by_type = {}
if nlp_metrics:
    nlp_default = nlp_metrics.get('default_argmax', {})
    nlp_by_type = by_type_rates_from_counts(nlp_default.get('by_type', {}))

if nlp_by_type:
    by_type_sources['NLP DistilBERT'] = nlp_by_type
else:
    by_type_sources['NLP DistilBERT (fallback)'] = nlp_recall_fallback

mlp_by_type = by_type_rates_from_counts(mlp_report.get('by_type', {}) if mlp_report else {})
if mlp_by_type:
    by_type_sources['MLP features'] = mlp_by_type

categories = [
    'benign (FPR)',
    'malicious_zeros',
    'malicious_random',
    'malicious_shuffle',
    'malicious_label',
]

by_type_df = pd.DataFrame(
    {
        name: [values.get(category, np.nan) for category in categories]
        for name, values in by_type_sources.items()
    },
    index=categories,
)
by_type_df


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
by_type_df.plot.bar(ax=ax, width=0.75)
ax.set_title('FPR em benignos e recall por tipo de ataque — DistilBERT vs MLP')
ax.set_ylabel('Taxa')
ax.set_ylim(0, 1.1)
ax.set_xticklabels(by_type_df.index, rotation=20, ha='right')
ax.grid(alpha=0.3, axis='y')
ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='50%')
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8, padding=2)
ax.legend(loc='center right')
plt.tight_layout()
out = OUT / 'plot_recall_by_attack.png'
plt.savefig(out, dpi=160, bbox_inches='tight')
out


## 7. Tabela visual por tipo de ataque

Tabela PNG com os valores exatos usados no gráfico anterior. Em `benign (FPR)`, menor é melhor; nos ataques, maior é melhor.


In [ ]:
if by_type_df.empty:
    print('Sem dados por tipo de ataque para plotar.')
else:
    table_df = by_type_df.copy()
    fig, ax = plt.subplots(figsize=(10, 2.8 + 0.25 * len(table_df)))
    ax.axis('off')
    table = ax.table(
        cellText=table_df.apply(lambda col: col.map(lambda x: '' if pd.isna(x) else f'{x:.3f}')).values,
        rowLabels=table_df.index,
        colLabels=table_df.columns,
        cellLoc='center',
        loc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.35)
    ax.set_title('FPR/recall por tipo de ataque — valores exatos', pad=16)
    plt.tight_layout()
    out = OUT / 'plot_detector_attack_table.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    display(out)


## 8. Foco em label flip

Este gráfico isola o ataque `malicious_label`, que tende a ser o caso mais difícil para fingerprint de pesos.


In [ ]:
label_row = by_type_df.loc[['malicious_label']].T.rename(columns={'malicious_label': 'label_flip_recall'})
if label_row.empty:
    print('Sem dados de malicious_label para plotar.')
else:
    fig, ax = plt.subplots(figsize=(7, 4))
    label_row.plot.bar(ax=ax, legend=False, color='#8c564b')
    ax.set_title('Recall em label flip — DistilBERT vs MLP')
    ax.set_ylabel('Recall')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(label_row.index, rotation=0)
    ax.grid(alpha=0.3, axis='y')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=9, padding=3)
    plt.tight_layout()
    out = OUT / 'plot_label_flip_recall.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    display(out)


## 9. Foco nas defesas aprendidas (`cc=6`, `cc=7`, `cc=8`, `cc=9`)

Comparação direta das defesas que usam detector treinado, separando os baselines `cc=2/cc=3`.


In [ ]:
learned_mask = summary['Defesa'].apply(lambda x: x.startswith(('cc=6', 'cc=7', 'cc=8', 'cc=9')))
learned_summary = summary.loc[learned_mask].set_index('Defesa')[['FPR_mean', 'FRR_mean']]
if learned_summary.empty:
    print('Sem CSVs de cc=6/cc=7/cc=8/cc=9 para plotar.')
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    learned_summary.plot.bar(ax=ax, color=['#3498db', '#e74c3c'], width=0.72)
    ax.set_title('Defesas com detector treinado: cc=6 vs cc=7 vs cc=8 vs cc=9')
    ax.set_ylabel('Taxa media nos ultimos 30 rounds')
    ax.set_ylim(0, max(0.30, float(learned_summary.max().max()) * 1.25))
    ax.set_xticklabels(learned_summary.index, rotation=15, ha='right')
    ax.grid(alpha=0.3, axis='y')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.4f', fontsize=8, padding=2)
    plt.tight_layout()
    out = OUT / 'plot_learned_defenses_fpr_frr.png'
    plt.savefig(out, dpi=160, bbox_inches='tight')
    display(out)


## 10. Sumário visual: barras agrupadas FPR/FRR (médias últimos 30 rounds)


In [ ]:
summary_plot = summary.set_index('Defesa')[['FPR_mean', 'FRR_mean']]
fig, ax = plt.subplots(figsize=(12, 5))
summary_plot.plot.bar(ax=ax, color=['#3498db', '#e74c3c'], width=0.75)
ax.set_title('Defesas em producao FL: FPR vs FRR — medias dos ultimos 30 rounds')
ax.set_ylabel('Taxa')
ax.set_ylim(0, max(summary_plot.max().max() * 1.20, 0.30))
ax.set_xticklabels(summary_plot.index, rotation=20, ha='right')
ax.grid(alpha=0.3, axis='y')
ax.legend(loc='upper right')
for container in ax.containers:
    ax.bar_label(container, fmt='%.4f', fontsize=8, padding=2)
plt.tight_layout()
out = OUT / 'plot_summary_fpr_frr.png'
plt.savefig(out, dpi=160, bbox_inches='tight')
print('\nRanking pelo criterio FPR+FRR (menor = melhor):')
print(summary[['Defesa', 'FPR_mean', 'FRR_mean', 'Score (FPR+FRR)']].to_string(index=False))
out


## Conclusões

1. Compare primeiro os detectores isolados (`plot_detector_metrics.png` e `plot_recall_by_attack.png`) para entender o comportamento do DistilBERT vs MLP.
2. Compare depois as defesas em produção (`plot_summary_fpr_frr.png` e `plot_learned_defenses_fpr_frr.png`), porque F1 alto no detector não garante FPR/FRR melhor no FL.
3. Use `cc=7` como baseline principal quando ele mantiver FPR baixo e FRR competitivo nos últimos 30 rounds.
4. Use `cc=8` como experimento comparativo: se `fpr_frr_results_8.csv` existir, ele aparece nos gráficos, mas só deve ser tratado como melhoria se reduzir FPR/FRR contra `cc=7`.
5. O gráfico de `label flip` deve ser lido separado, porque esse ataque costuma ser o limite dos detectores baseados só em fingerprint dos pesos.

Detalhes científicos completos em `MONZA_RESULTS.md`.
